# Cross-Section Design with Catalog Components

This notebook demonstrates the integration of the component catalog system with cross-section design.

**Workflow:**
1. Select concrete and reinforcement from catalogs
2. Create cross-section geometry (rectangular, T-shape, I-shape)
3. Define reinforcement layers with catalog components
4. Assemble complete cross-section
5. Visualize and analyze

**Key Benefits:**
- Product traceability (product IDs, manufacturers)
- Automatic design value calculations (f_cd, f_td with safety factors)
- Consistent material properties from validated catalog
- Ready for Phase 3 moment-curvature analysis

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Component catalog system
from bmcs_cross_section.cs_components import (
    create_steel_rebar_catalog,
    create_concrete_catalog,
    get_concrete_by_class,
    SteelRebarComponent,
    CarbonBarComponent,
    TextileReinforcementComponent,
)

# Cross-section design (modern API)
from bmcs_cross_section.cs_design import (
    RectangularShape,
    TShape,
    IShape,
    ReinforcementLayer,
    ReinforcementLayout,
    CrossSection,
    create_symmetric_reinforcement,
)

# Configure plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ All imports successful")

## 2. Example 1: Rectangular Beam with Steel Reinforcement

Create a simple rectangular beam using catalog components.

### Step 1: Select Components from Catalog

In [ ]:
# Select concrete grade
concrete_c30 = get_concrete_by_class('C30/37')

# Select steel rebars
steel_d20 = SteelRebarComponent(nominal_diameter=20, grade='B500B')
steel_d12 = SteelRebarComponent(nominal_diameter=12, grade='B500B')

print("Selected Components:")
print("=" * 60)
print(f"\nConcrete: {concrete_c30.name}")
print(f"  Product ID: {concrete_c30.product_id}")
print(f"  f_ck = {concrete_c30.f_ck} MPa (characteristic)")
print(f"  f_cm = {concrete_c30.f_cm} MPa (mean)")
print(f"  f_cd = {concrete_c30.f_cd:.2f} MPa (design)")
print(f"  E_cm = {concrete_c30.E_cm} MPa")

print(f"\nBottom Reinforcement: {steel_d20.name}")
print(f"  Product ID: {steel_d20.product_id}")
print(f"  Diameter: {steel_d20.nominal_diameter} mm")
print(f"  Area per bar: {steel_d20.area:.2f} mm²")
print(f"  f_tk = {steel_d20.f_tk} MPa (characteristic)")
print(f"  f_td = {steel_d20.f_td:.1f} MPa (design)")

print(f"\nTop Reinforcement: {steel_d12.name}")
print(f"  Product ID: {steel_d12.product_id}")
print(f"  Diameter: {steel_d12.nominal_diameter} mm")
print(f"  Area per bar: {steel_d12.area:.2f} mm²")

### Step 2: Create Cross-Section Geometry

In [ ]:
# Define rectangular shape
shape = RectangularShape(
    b=300,  # width [mm]
    h=500   # height [mm]
)

print(f"Cross-Section Geometry: Rectangular")
print(f"  Width:  {shape.b} mm")
print(f"  Height: {shape.h} mm")
print(f"  Area:   {shape.area:.0f} mm²")

### Step 3: Define Reinforcement Layers Using Catalog Components

In [ ]:
# Bottom layer: 4 bars of D20
n_bars_bottom = 4
As_bottom = steel_d20.area * n_bars_bottom

bottom_layer = ReinforcementLayer(
    z=450,  # distance from top [mm]
    A_s=As_bottom,
    material=steel_d20.matmod  # Material model from catalog component
)

# Top layer: 2 bars of D12
n_bars_top = 2
As_top = steel_d12.area * n_bars_top

top_layer = ReinforcementLayer(
    z=50,  # distance from top [mm]
    A_s=As_top,
    material=steel_d12.matmod  # Material model from catalog component
)

# Create reinforcement layout
reinforcement = ReinforcementLayout(
    layers=[bottom_layer, top_layer]
)

print("Reinforcement Layout:")
print("=" * 60)
print(f"\nBottom Layer (z = {bottom_layer.z} mm from top):")
print(f"  {n_bars_bottom} × D{steel_d20.nominal_diameter}")
print(f"  Total area: {As_bottom:.0f} mm²")
print(f"  Component: {steel_d20.product_id}")
print(f"  Distance from bottom: {shape.h - bottom_layer.z} mm")

print(f"\nTop Layer (z = {top_layer.z} mm from top):")
print(f"  {n_bars_top} × D{steel_d12.nominal_diameter}")
print(f"  Total area: {As_top:.0f} mm²")
print(f"  Component: {steel_d12.product_id}")

### Step 4: Assemble Complete Cross-Section

In [ ]:
# Create cross-section with catalog components
cross_section = CrossSection(
    shape=shape,
    concrete=concrete_c30.matmod,  # Concrete material from catalog
    reinforcement=reinforcement
)

print("✓ Cross-Section Assembly Complete")
print("=" * 60)
print(f"\nConcrete: {concrete_c30.name}")
print(f"  Design strength: f_cd = {concrete_c30.f_cd:.2f} MPa")
print(f"\nReinforcement:")
print(f"  Bottom: {n_bars_bottom} × D{steel_d20.nominal_diameter} = {As_bottom:.0f} mm²")
print(f"  Top:    {n_bars_top} × D{steel_d12.nominal_diameter} = {As_top:.0f} mm²")
print(f"  Total:  {As_bottom + As_top:.0f} mm²")
print(f"\nReinforcement Ratio: {(As_bottom + As_top) / shape.area * 100:.2f}%")

### Step 5: Visualize Cross-Section

In [ ]:
# Plot cross-section
cross_section.plot_cross_section()
plt.title(f'Rectangular Beam: {shape.b}×{shape.h} mm\n' + 
          f'{concrete_c30.name}, {n_bars_bottom}×D{steel_d20.nominal_diameter} + {n_bars_top}×D{steel_d12.nominal_diameter}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Example 2: T-Beam with High-Strength Concrete

Demonstrate T-shaped cross-section with higher concrete grade.

In [ ]:
# Select high-strength concrete
concrete_c50 = get_concrete_by_class('C50/60')

# Select reinforcement
steel_d25 = SteelRebarComponent(nominal_diameter=25, grade='B500B')
steel_d16 = SteelRebarComponent(nominal_diameter=16, grade='B500B')

# Create T-shape
t_shape = TShape(
    b_f=800,  # flange width [mm]
    h_f=150,  # flange height [mm]
    b_w=300,  # web width [mm]
    h=600     # total height [mm]
)

# Bottom reinforcement: 6 × D25
n_bars_bottom_t = 6
As_bottom_t = steel_d25.area * n_bars_bottom_t
bottom_layer_t = ReinforcementLayer(
    z=550,  # 50mm from bottom
    A_s=As_bottom_t,
    material=steel_d25.matmod
)

# Top reinforcement: 4 × D16
n_bars_top_t = 4
As_top_t = steel_d16.area * n_bars_top_t
top_layer_t = ReinforcementLayer(
    z=50,
    A_s=As_top_t,
    material=steel_d16.matmod
)

# Assemble T-beam
reinforcement_t = ReinforcementLayout(layers=[bottom_layer_t, top_layer_t])
t_beam = CrossSection(
    shape=t_shape,
    concrete=concrete_c50.matmod,
    reinforcement=reinforcement_t
)

print("T-Beam Configuration:")
print("=" * 60)
print(f"\nConcrete: {concrete_c50.name}")
print(f"  f_ck = {concrete_c50.f_ck} MPa")
print(f"  f_cd = {concrete_c50.f_cd:.2f} MPa")
print(f"\nGeometry:")
print(f"  Flange: {t_shape.b_f} × {t_shape.h_f} mm")
print(f"  Web:    {t_shape.b_w} × {t_shape.h - t_shape.h_f} mm")
print(f"  Total height: {t_shape.h} mm")
print(f"\nReinforcement:")
print(f"  Bottom: {n_bars_bottom_t} × D{steel_d25.nominal_diameter} ({steel_d25.product_id})")
print(f"  Top:    {n_bars_top_t} × D{steel_d16.nominal_diameter} ({steel_d16.product_id})")

In [ ]:
# Visualize T-beam
t_beam.plot_cross_section()
plt.title(f'T-Beam: Flange {t_shape.b_f}×{t_shape.h_f} mm, Web {t_shape.b_w}×{t_shape.h-t_shape.h_f} mm\n' +
          f'{concrete_c50.name}, {n_bars_bottom_t}×D{steel_d25.nominal_diameter} + {n_bars_top_t}×D{steel_d16.nominal_diameter}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Example 3: Comparison of Different Concrete Grades

Show how different concrete grades affect cross-section design.

In [ ]:
# Create identical cross-sections with different concrete grades
concrete_grades = ['C25/30', 'C30/37', 'C40/50', 'C50/60']
cross_sections = []

# Same geometry and reinforcement
common_shape = RectangularShape(b=300, h=500)
steel_d16_common = SteelRebarComponent(nominal_diameter=16, grade='B500B')

# Bottom: 4×D16, Top: 2×D16
As_bottom_common = steel_d16_common.area * 4
As_top_common = steel_d16_common.area * 2

for grade in concrete_grades:
    concrete = get_concrete_by_class(grade)
    
    bottom = ReinforcementLayer(z=450, A_s=As_bottom_common, material=steel_d16_common.matmod)
    top = ReinforcementLayer(z=50, A_s=As_top_common, material=steel_d16_common.matmod)
    reinf = ReinforcementLayout(layers=[bottom, top])
    
    cs = CrossSection(
        shape=common_shape,
        concrete=concrete.matmod,
        reinforcement=reinf
    )
    
    cross_sections.append((grade, concrete, cs))

# Display comparison
print("Concrete Grade Comparison:")
print("=" * 80)
print(f"{'Grade':<12} {'f_ck (MPa)':<12} {'f_cm (MPa)':<12} {'f_cd (MPa)':<12} {'E_cm (MPa)':<12}")
print("=" * 80)
for grade, concrete, _ in cross_sections:
    print(f"{grade:<12} {concrete.f_ck:<12} {concrete.f_cm:<12} {concrete.f_cd:<12.2f} {concrete.E_cm:<12}")
print("=" * 80)
print(f"\nAll sections have identical geometry: {common_shape.b}×{common_shape.h} mm")
print(f"All sections have identical reinforcement: 4×D16 + 2×D16")

## 5. Example 4: Using Symmetric Reinforcement Helper

Demonstrate the convenience function for symmetric reinforcement.

In [ ]:
# Select components
concrete_c35 = get_concrete_by_class('C35/45')
steel_d20_sym = SteelRebarComponent(nominal_diameter=20, grade='B500B')

# Create shape
shape_sym = RectangularShape(b=400, h=600)

# Create symmetric reinforcement using helper function
n_bars = 5
As_layer = steel_d20_sym.area * n_bars

reinforcement_sym = create_symmetric_reinforcement(
    A_s=As_layer,
    material=steel_d20_sym.matmod,
    h=shape_sym.h,
    cover=50  # cover distance [mm]
)

# Assemble
cs_sym = CrossSection(
    shape=shape_sym,
    concrete=concrete_c35.matmod,
    reinforcement=reinforcement_sym
)

print("Symmetric Reinforcement Configuration:")
print("=" * 60)
print(f"\nConcrete: {concrete_c35.name}")
print(f"Cross-section: {shape_sym.b} × {shape_sym.h} mm")
print(f"\nReinforcement (symmetric):")
print(f"  {n_bars} × D{steel_d20_sym.nominal_diameter} at top (z=50 mm)")
print(f"  {n_bars} × D{steel_d20_sym.nominal_diameter} at bottom (z=550 mm)")
print(f"  Component: {steel_d20_sym.product_id}")
print(f"  Total area per layer: {As_layer:.0f} mm²")
print(f"  Total reinforcement: {2 * As_layer:.0f} mm²")

In [ ]:
# Visualize
cs_sym.plot_cross_section()
plt.title(f'Symmetric Reinforcement: {shape_sym.b}×{shape_sym.h} mm\n' +
          f'{concrete_c35.name}, 2 × {n_bars}×D{steel_d20_sym.nominal_diameter}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Bill of Materials from Catalog

Generate material list with traceability to catalog components.

In [ ]:
# Function to generate BOM for a cross-section
def generate_bom(cross_section_info, length_m=1.0):
    """
    Generate bill of materials for a cross-section.
    
    Parameters:
    -----------
    cross_section_info : dict
        Dictionary with 'concrete', 'shape', 'reinforcement' info
    length_m : float
        Length of element in meters
    """
    shape = cross_section_info['shape']
    concrete = cross_section_info['concrete']
    reinforcement_list = cross_section_info['reinforcement']
    
    # Calculate volumes/quantities
    concrete_volume_m3 = shape.area * length_m * 1000 / 1e9  # mm² → m³
    
    print("BILL OF MATERIALS")
    print("=" * 80)
    print(f"\nElement: {cross_section_info['name']}")
    print(f"Length: {length_m} m")
    print(f"Cross-section: {shape.b} × {shape.h} mm")
    print("\n" + "=" * 80)
    
    # Concrete
    print(f"\n1. CONCRETE")
    print(f"   Product: {concrete.name}")
    print(f"   Product ID: {concrete.product_id}")
    print(f"   Grade: {concrete.strength_class}")
    print(f"   f_ck: {concrete.f_ck} MPa")
    print(f"   Volume: {concrete_volume_m3:.3f} m³")
    
    # Reinforcement
    print(f"\n2. REINFORCEMENT")
    for i, reinf_info in enumerate(reinforcement_list, 1):
        component = reinf_info['component']
        n_bars = reinf_info['n_bars']
        layer_name = reinf_info['layer_name']
        z_pos = reinf_info['z']
        
        # Calculate length and weight
        total_length_m = n_bars * length_m
        weight_kg = component.area * length_m * 1000 * 7.85e-6 * n_bars  # steel density 7.85 kg/dm³
        
        print(f"\n   2.{i} {layer_name}")
        print(f"       Product: {component.name}")
        print(f"       Product ID: {component.product_id}")
        print(f"       Diameter: {component.nominal_diameter} mm")
        print(f"       Quantity: {n_bars} bars")
        print(f"       Position: z = {z_pos} mm from top ({shape.h - z_pos} mm from bottom)")
        print(f"       Total length: {total_length_m:.1f} m")
        print(f"       Estimated weight: {weight_kg:.2f} kg")
    
    print("\n" + "=" * 80)

# Example BOM for rectangular beam
bom_info = {
    'name': 'Rectangular Beam',
    'shape': shape,
    'concrete': concrete_c30,
    'reinforcement': [
        {
            'component': steel_d20,
            'n_bars': n_bars_bottom,
            'layer_name': 'Bottom layer',
            'z': bottom_layer.z
        },
        {
            'component': steel_d12,
            'n_bars': n_bars_top,
            'layer_name': 'Top layer',
            'z': top_layer.z
        }
    ]
}

generate_bom(bom_info, length_m=6.0)

## 7. Summary: Benefits of Catalog Integration

### Key Advantages:

1. **Product Traceability**
   - Every material has a product ID and manufacturer
   - Easy to generate specifications and material orders
   - Audit trail for design decisions

2. **Automatic Safety Factors**
   - Design values (f_cd, f_td) automatically calculated
   - Consistent with EC2 requirements (γ_c=1.5, γ_s=1.15)
   - No manual calculations or errors

3. **Validated Material Properties**
   - All properties from standardized catalogs
   - Consistent across entire design
   - Easy to update when new products available

4. **Seamless Integration**
   - Component.matmod provides material model
   - Works with existing cs_design infrastructure
   - Ready for Phase 3 moment-curvature analysis

5. **Design Automation**
   - Quick comparison of alternatives
   - Parametric studies with catalog products
   - Optimization with real product constraints

### Next Steps:
- Phase 3: Moment-curvature analysis with catalog components
- Automatic component selection based on requirements
- Cost optimization using catalog data
- Export to design tools and BIM systems

In [ ]:
print("✓ Notebook complete: Cross-section design with catalog components")
print("  - Rectangular, T-beam, and symmetric reinforcement examples")
print("  - Full integration with component catalog")
print("  - Bill of materials generation")
print("  - Ready for Phase 3 analysis")